# IMPORTAZIONE DELLE LIBRERIE


In [64]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import pandas as pd
import os
import requests

In [65]:
os.getcwd()

'c:\\Users\\eliav\\Desktop\\Road_accidents_analysis\\Road_accidents_analysis_FIXED'

In [66]:
if not os.path.exists("Road_accidents_analysis_FIXED/FIXED_fetching data"):
    os.mkdir("FIXED_fetching data")

# SCRAPING DA SITUAS

In [67]:
url_situas = "https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2020-12-31"

resp_situas = requests.get (url_situas) 

print("Risposta alla richiesta:", resp_situas.status_code)

Risposta alla richiesta: 200


In [68]:
# =============================================================================
# LIBRERIE
# =============================================================================

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from pathlib import Path
import time
import os

# =============================================================================
# COSTANTI
# =============================================================================

# URL del report SITUAS
BASE_URL = "https://situas.istat.it/web/#/territorio/body"

# ID del report
REPORT_ID = 74

# Intervallo di anni da scaricare
ANNO_INIZIO = 2001
ANNO_FINE = 2024

# Timeout massimo delle attese esplicite
TIMEOUT = 120

# Cartella (relativa al progetto) in cui salvare i CSV
DOWNLOAD_DIR = Path("FIXED_fetching data") 
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# =============================================================================
# CONFIGURAZIONE DI CHROME
# =============================================================================

options = webdriver.ChromeOptions()

prefs = {
    # Cartella di download
    "download.default_directory": str(DOWNLOAD_DIR.resolve()),

    # Disabilita la richiesta di conferma del download
    "download.prompt_for_download": False,

    # Utilizza sempre la cartella specificata
    "download.directory_upgrade": True
}

options.add_experimental_option("prefs", prefs)

# Avvio del browser
driver = webdriver.Chrome(options=options)

# Oggetto utilizzato per le attese esplicite
wait = WebDriverWait(driver, TIMEOUT)

# =============================================================================
# SELETTORI
# =============================================================================

EXPORT_BUTTON = (
    By.ID,
    "dati-report-export-btn"
)

CSV_BUTTON = (
    By.CSS_SELECTOR,
    'button[title="Scarica dati in formato CSV"]'
)

# =============================================================================
# FUNZIONI
# =============================================================================

def apri_report(anno):
    """
    Apre il report relativo all'anno richiesto.
    """

    url = (
        f"{BASE_URL}"
        f"?id={REPORT_ID}"
        f"&dateFrom={anno}-12-31"
    )

    driver.get(url)

    # Attende il caricamento completo della pagina
    wait.until(
        lambda d: d.execute_script(
            "return document.readyState"
        ) == "complete"
    )


def clicca_esporta():
    """
    Apre il menu di esportazione.
    """

    export = wait.until(
        EC.element_to_be_clickable(EXPORT_BUTTON)
    )

    driver.execute_script(
        "arguments[0].scrollIntoView({block:'center'});",
        export
    )

    driver.execute_script(
        "arguments[0].click();",
        export
    )


def clicca_csv():
    """
    Clicca sul pulsante CSV ed avvia il download.
    """

    csv = wait.until(
        EC.element_to_be_clickable(CSV_BUTTON)
    )

    driver.execute_script(
        "arguments[0].click();",
        csv
    )

    # Piccola attesa affinché Chrome inizi il download
    time.sleep(3)


def attendi_download(file_prima):
    """
    Attende che venga scaricato un nuovo file CSV.

    Parameters
    ----------
    file_prima : set
        Insieme dei file presenti prima del download.

    Returns
    -------
    Path
        Percorso del nuovo CSV scaricato.
    """

    # Attende che inizi il download
    wait.until(
        lambda d:
            len(list(DOWNLOAD_DIR.glob("*.crdownload"))) > 0
            or
            len(set(DOWNLOAD_DIR.glob("*.csv")) - file_prima) > 0
    )

    # Attende la fine del download
    wait.until(
        lambda d:
            len(list(DOWNLOAD_DIR.glob("*.crdownload"))) == 0
    )

    # Individua il nuovo CSV scaricato
    nuovi_file = list(
        set(DOWNLOAD_DIR.glob("*.csv")) - file_prima
    )

    return nuovi_file[0]


def rinomina_csv(file_csv, anno):
    """
    Rinomina il CSV appena scaricato.
    """

    nuovo_nome = DOWNLOAD_DIR / f"situas_{anno}.csv"

    os.replace(
        file_csv,
        nuovo_nome
    )

# =============================================================================
# DOWNLOAD DEI REPORT
# =============================================================================

for anno in range(ANNO_INIZIO, ANNO_FINE + 1):

    print(f"\n========== {anno} ==========")

    # Memorizza i file presenti prima del download
    file_prima = set(DOWNLOAD_DIR.glob("*.csv"))

    print("Apro il report...")
    apri_report(anno)

    print("Apro il menu Esporta...")
    clicca_esporta()

    print("Avvio il download del CSV...")
    clicca_csv()

    print("Attendo il completamento del download...")
    file_csv = attendi_download(file_prima)

    print("Rinomino il file...")
    rinomina_csv(file_csv, anno)

    print(f"✔ Report {anno} scaricato correttamente")

    # Piccola pausa prima dell'anno successivo
    time.sleep(2)

# =============================================================================
# CHIUSURA DEL BROWSER
# =============================================================================

driver.quit()

print("\nDownload completato con successo!")


========== 2001 ==========
Apro il report...
Apro il menu Esporta...
Avvio il download del CSV...
Attendo il completamento del download...
Rinomino il file...
✔ Report 2001 scaricato correttamente

========== 2002 ==========
Apro il report...
Apro il menu Esporta...
Avvio il download del CSV...
Attendo il completamento del download...
Rinomino il file...
✔ Report 2002 scaricato correttamente

========== 2003 ==========
Apro il report...
Apro il menu Esporta...
Avvio il download del CSV...
Attendo il completamento del download...
Rinomino il file...
✔ Report 2003 scaricato correttamente

========== 2004 ==========
Apro il report...
Apro il menu Esporta...
Avvio il download del CSV...
Attendo il completamento del download...
Rinomino il file...
✔ Report 2004 scaricato correttamente

========== 2005 ==========
Apro il report...
Apro il menu Esporta...
Avvio il download del CSV...
Attendo il completamento del download...
Rinomino il file...
✔ Report 2005 scaricato correttamente

=========

In [ ]:
# =============================================================================
# LIBRERIE
# =============================================================================

import pandas as pd
from pathlib import Path

# =============================================================================
# LETTURA E UNIONE DEI CSV SITUAS
# =============================================================================

# Cartella contenente i CSV scaricati
DOWNLOAD_DIR = Path("FIXED_fetching data")

# Elenco dei file CSV
file_csv = sorted(DOWNLOAD_DIR.glob("*.csv"))

# Lista dei DataFrame
lista_df = []

# Lettura dei file
for file in file_csv:

    print(f"Lettura di {file.name}")

    # Lettura del CSV (aggiungi sep=';' se necessario)
    df = pd.read_csv(file, sep=";")

    # Estrae l'anno dal nome del file
    # Es.: situas_2001.csv -> 2001
    anno = int(file.stem.split("_")[1])

    df["TIME_PERIOD"]=anno #questo mi sarà utile per la merge con istat, perchè così dovrei avere comune univoco per annno
                                #univoco, e quindi queste due colonne verranno joinate con il df istat

    # Aggiunge il DataFrame alla lista
    lista_df.append(df)

# Unione di tutti i DataFrame
df_situas = pd.concat(
    lista_df,
    ignore_index=True
)

# Controllo
print(df_situas.shape)
display(df_situas.head(20))

Lettura di situas_2001.csv
Lettura di situas_2002.csv
Lettura di situas_2003.csv
Lettura di situas_2004.csv
Lettura di situas_2005.csv
Lettura di situas_2006.csv
Lettura di situas_2007.csv
Lettura di situas_2008.csv
Lettura di situas_2009.csv
Lettura di situas_2010.csv
Lettura di situas_2011.csv
Lettura di situas_2012.csv
Lettura di situas_2013.csv
Lettura di situas_2014.csv
Lettura di situas_2015.csv
Lettura di situas_2016.csv
Lettura di situas_2017.csv
Lettura di situas_2018.csv
Lettura di situas_2019.csv
Lettura di situas_2020.csv
Lettura di situas_2021.csv
Lettura di situas_2022.csv
Lettura di situas_2023.csv
Lettura di situas_2024.csv
(192731, 18)


,Codice Ripartizione geografica,Codice Regione,Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente),TIME_PERIOD,Codice Provincia (Storico)
0,1,1,1,1001,1001,Agliè,NaN,TO,0,0,2574.0,2001,"13,1462",2001,2557,2001,2001,NaN
1,1,1,1,1002,1002,Airasca,NaN,TO,0,0,3554.0,2001,"15,7393",2001,3543,2001,2001,NaN
2,1,1,1,1003,1003,Ala di Stura,NaN,TO,0,0,479.0,2001,"46,3315",2001,480,2001,2001,NaN
3,1,1,1,1004,1004,Albiano d'Ivrea,NaN,TO,0,0,1696.0,2001,"11,7315",2001,1687,2001,2001,NaN
4,1,1,1,1005,1005,Alice Superiore,NaN,TO,0,0,616.0,2001,"7,3796",2001,619,2001,2001,NaN
5,1,1,1,1006,1006,Almese,NaN,TO,0,0,5658.0,2001,"17,8756",2001,5648,2001,2001,NaN
6,1,1,1,1007,1007,Alpette,NaN,TO,0,0,300.0,2001,"5,6261",2001,293,2001,2001,NaN
7,1,1,1,1008,1008,Alpignano,NaN,TO,0,0,16648.0,2001,"11,9193",2001,16647,2001,2001,NaN
8,1,1,1,1009,1009,Andezeno,NaN,TO,0,0,1705.0,2001,"7,4861",2001,1712,2001,2001,NaN
9,1,1,1,1010,1010,Andrate,NaN,TO,0,0,476.0,2001,"9,3085",2001,474,2001,2001,NaN


In [2]:
df_situas.nunique()

Codice Ripartizione geografica        5
Codice Regione                       20
Codice Provincia/Uts                125
Codice Comune (alfanumerico)       8578
Codice Comune (numerico)           8578
Comune                             8249
Comune (dizione straniera)          128
Sigla automobilistica               110
Capoluogo di Provincia/Uts            2
Capoluogo di Regione                  2
Popolazione legale                10254
Anno Censimento                       3
Superficie (Kmq)                  17391
Anno (Superficie)                    24
Popolazione residente             28000
Anno (Popolazione residente)         24
TIME_PERIOD                          24
Codice Provincia (Storico)          111
dtype: int64

In [3]:
def controlla_dataframe(df):

    for colonna in df.columns:

        print("=" * 70)
        print(f"COLONNA: {colonna}")
        print("=" * 70)

        print(f"Tipo: {df[colonna].dtype}")
        print(f"Valori mancanti: {df[colonna].isna().sum()}")
        print(f"Valori unici: {df[colonna].nunique()}")

        print("\nValori più frequenti:")

        print(
            df[colonna]
            .value_counts(dropna=False)
            .head(10)
        )

        print("\n")

In [4]:
controlla_dataframe(df_situas)

COLONNA: Codice Ripartizione geografica
Tipo: int64
Valori mancanti: 0
Valori unici: 5

Valori più frequenti:
Codice Ripartizione geografica
1    72933
4    42903
2    34762
3    23721
5    18412
Name: count, dtype: int64


COLONNA: Codice Regione
Tipo: int64
Valori mancanti: 0
Valori unici: 20

Valori più frequenti:
Codice Regione
3     36749
1     28775
5     13806
15    13212
18     9777
19     9364
12     9072
20     9048
8      8122
4      7621
Name: count, dtype: int64


COLONNA: Codice Provincia/Uts
Tipo: int64
Valori mancanti: 0
Valori unici: 125

Valori più frequenti:
Codice Provincia/Uts
4     5982
16    5841
17    4935
22    4837
6     4540
18    4526
1     4410
65    3792
13    3751
78    3681
Name: count, dtype: int64


COLONNA: Codice Comune (alfanumerico)
Tipo: int64
Valori mancanti: 0
Valori unici: 8578

Valori più frequenti:
Codice Comune (alfanumerico)
1001    24
1002    24
1003    24
1004    24
1006    24
1007    24
1008    24
1009    24
1010    24
1011    24
Name: c

In [5]:
#dalle informazioni di df_situas ho visto che ci sono 24 comuni senza nome, li cerco
df_situas[df_situas["Comune"].isna()]

,Codice Ripartizione geografica,Codice Regione,Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente),TIME_PERIOD,Codice Provincia (Storico)
167,1,1,1,1168,1168,NaN,NaN,TO,0,0,7761.0,2001,"24,6422",2001,7749,2001,2001,NaN
8269,1,1,1,1168,1168,NaN,NaN,TO,0,0,7761.0,2001,"24,6422",2002,7777,2002,2002,NaN
16371,1,1,1,1168,1168,NaN,NaN,TO,0,0,7761.0,2001,"24,6422",2003,7822,2003,2003,NaN
24471,1,1,1,1168,1168,NaN,NaN,TO,0,0,7761.0,2001,"24,6422",2004,7838,2004,2004,NaN
32572,1,1,1,1168,1168,NaN,NaN,TO,0,0,7761.0,2001,"24,6422",2005,7815,2005,2005,NaN
40673,1,1,1,1168,1168,NaN,NaN,TO,0,0,7761.0,2001,"24,6422",2006,7798,2006,2006,NaN
48774,1,1,1,1168,1168,NaN,NaN,TO,0,0,7761.0,2001,"24,6422",2007,7866,2007,2007,NaN
56875,1,1,1,1168,1168,NaN,NaN,TO,0,0,7761.0,2001,"24,6422",2008,7873,2008,2008,NaN
64976,1,1,1,1168,1168,NaN,NaN,TO,0,0,7761.0,2001,"24,6422",2009,7859,2009,2009,NaN
73076,1,1,1,1168,1168,NaN,NaN,TO,0,0,7761.0,2001,"24,6422",2010,7986,2010,2010,NaN


In [6]:
#Sostituisco i valori letti come NaN con il nome vero del comune "None" solo che
#devo aggiungere un "." sennò lo leggerebbe come valore nullo
df_situas["Comune"] = df_situas["Comune"].fillna("None.")

In [7]:
df_situas["Comune"].isna().sum()

np.int64(0)

In [8]:
#se non esiste la cartella df raw me la faccio creare
if not os.path.exists("Road_accidents_analysis_FIXED/Df_raw"):
    os.mkdir("Df_raw")

NameError: name 'os' is not defined

In [9]:
#e ci salvo dentro il mio df situas grezzo

df_situas.to_csv("Df_raw/df_situas_raw.csv", index=False)

# RICHIAMO API per ISTAT

Tramite richiesta API scarico il dataset di istat 

In [ ]:
# url_istat = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"

# http_header = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}

# resp_istat = requests.get(url_istat, headers= http_header )

# print("Risposta alla richiesta:", resp_istat.status_code)

Risposta alla richiesta: 200


In [ ]:
# #Impongo condizione per cui se non esiste il file incidenti_istat venga creato e si scriva la risposta all' API (il dataset)
# if not os.path.exists("Df_raw/df_istat_raw.csv"):
#     istat_incidents = open("Df_raw/df_istat_raw.csv", "w", encoding="utf-8") #utf-8 serve per avere i caratteri corretti
#     istat_incidents.write(resp_istat.text)
#     istat_incidents.close()

In [ ]:
# df_istat_raw= pd.read_csv("Df_raw/df_istat_raw.csv")
# df_istat_raw

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
573547,IT1:41_983(1.0),A,111107,ROADACC,9,2020,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
573548,IT1:41_983(1.0),A,111107,ROADACC,9,2021,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
573549,IT1:41_983(1.0),A,111107,ROADACC,9,2022,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
573550,IT1:41_983(1.0),A,111107,ROADACC,9,2023,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# df_istat_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          573552 non-null  str    
 1   FREQ              573552 non-null  str    
 2   REF_AREA          573552 non-null  int64  
 3   DATA_TYPE         573552 non-null  str    
 4   RESULT            573552 non-null  str    
 5   TIME_PERIOD       573552 non-null  int64  
 6   OBS_VALUE         573552 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64(3), str(4)

In [ ]:
# controlla_dataframe(df_istat_raw)

COLONNA: DATAFLOW
Tipo: str
Valori mancanti: 0
Valori unici: 1

Valori più frequenti:
DATAFLOW
IT1:41_983(1.0)    573552
Name: count, dtype: int64


COLONNA: FREQ
Tipo: str
Valori mancanti: 0
Valori unici: 1

Valori più frequenti:
FREQ
A    573552
Name: count, dtype: int64


COLONNA: REF_AREA
Tipo: int64
Valori mancanti: 0
Valori unici: 8578

Valori più frequenti:
REF_AREA
1001    72
1002    72
1003    72
1004    72
1006    72
1008    72
1009    72
1010    72
1012    72
1013    72
Name: count, dtype: int64


COLONNA: DATA_TYPE
Tipo: str
Valori mancanti: 0
Valori unici: 2

Valori più frequenti:
DATA_TYPE
KILLINJ    382368
ROADACC    191184
Name: count, dtype: int64


COLONNA: RESULT
Tipo: str
Valori mancanti: 0
Valori unici: 3

Valori più frequenti:
RESULT
F    191184
M    191184
9    191184
Name: count, dtype: int64


COLONNA: TIME_PERIOD
Tipo: int64
Valori mancanti: 0
Valori unici: 24

Valori più frequenti:
TIME_PERIOD
2002    24306
2003    24306
2001    24303
2005    24303
2006    24